In [1]:
import ollama
import chromadb

In [2]:
# Load ChromaDB collections
client = chromadb.PersistentClient(path="./chroma_db_improved")

collection_global = client.get_or_create_collection(name="global_summaries")
collection_grouped = client.get_or_create_collection(name="grouped_summaries")
collection_raw = client.get_or_create_collection(name="raw_transactions")
MODEL = "mistral:7b"

print(f"  global_summaries: {collection_global.count()} chunks")
print(f"  grouped_summaries: {collection_grouped.count()} chunks")
print(f"  raw_transactions: {collection_raw.count()} chunks")

  global_summaries: 12 chunks
  grouped_summaries: 58 chunks
  raw_transactions: 1668 chunks


In [3]:
def classify_query(query):
    q = query.lower()
    if "month" in q or "season" in q: return "monthly"
    elif "year" in q or "trend" in q or "over time" in q: return "yearly"
    elif "subcategor" in q or "sub-categor" in q: return "subcategory"
    elif "discount" in q: return "discount"
    elif "state" in q or "city" in q: return "geography"
    elif "category" in q: return "category"
    elif "region" in q: return "region"
    else: return "general"

def retrieve_context(query, num_global=3, num_grouped=2, num_raw=0):
    qtype = classify_query(query)
    global_results = collection_global.query(query_texts=[query], n_results=num_global)
    
    if qtype == "monthly":
        grouped_results = collection_grouped.query(query_texts=[query], n_results=4, where={"fact_type": {"$eq": "metric"}})
    elif qtype == "geography":
        grouped_results = collection_grouped.query(query_texts=[query], n_results=4, where={"layer": {"$eq": "grouped"}})
    else:
        grouped_results = collection_grouped.query(query_texts=[query], n_results=num_grouped, where={"layer": {"$eq": "grouped"}})
    
    global_docs = global_results['documents'][0] if global_results['documents'] else []
    grouped_docs = grouped_results['documents'][0] if grouped_results['documents'] else []
    combined = []
    if global_docs: combined.extend(global_docs)
    if grouped_docs: combined.extend(grouped_docs)
    context = "\n\n".join(combined)
    if len(context) > 2000: context = context[:2000] + "\n... [context truncated]"
    return context, len(global_docs), len(grouped_docs), 0

In [4]:
def rag_query(query, model=MODEL):
    context, n_global, n_grouped, n_raw = retrieve_context(query)
    if not context or (n_global == 0 and n_grouped == 0):
        return "No relevant data found in knowledge base."
    
    prompt = f"""You are a sales analytics expert. Answer PRECISELY and CONCISELY.

DATA (in priority order):
{context}

ANSWER FORMAT:
- Use bullet points for lists
- Include exact numbers and dollar signs ($)
- One sentence explanation maximum
- NO estimates or approximations
- NO hedging language

QUESTION: {query}

ANSWER:"""
    
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=False,
        options={
            "num_predict": 150,
            "temperature": 0.1,
            "top_k": 40,
            "top_p": 0.9
        }
    )
    
    answer = response['message']['content']
    return answer, {
        'global_chunks': n_global,
        'grouped_chunks': n_grouped,
        'raw_chunks': 0,
        'context_size': len(context),
        'model': model,
        'query_type': classify_query(query)
    }

In [6]:
test_query = "What is the sales trend over the 4-year period?"
print(f"Q: {test_query}")
answer, metadata = rag_query(test_query, model=MODEL)
print(f"A: {answer}")
print(f"Chunks: {metadata['global_chunks']} global + {metadata['grouped_chunks']} grouped | Type: {metadata['query_type']}\n")

Q: What is the sales trend over the 4-year period?
A:  Sales trend over the 4-year period (2014-2017):
- 2014: $484,247.50 sales
- 2015: $470,532.51 sales (decrease by $7,715)
- 2016: $609,205.60 sales (increase by $138,673)
- 2017: $2,297,200.86 sales (increase by $1,697,994)
Chunks: 3 global + 2 grouped | Type: yearly



In [7]:
queries = {
    "TREND ANALYSIS": [
        "Which months show the highest sales? Is there seasonality?",
        "How has profit margin changed over time?"
    ],
    "CATEGORY ANALYSIS": [
        "Which product category generates the most revenue?",
        "What sub-categories have the highest profit margins?",
        "Which products are frequently sold at a discount?"
    ],
    "REGIONAL ANALYSIS": [
        "Which region has the best sales performance?",
        "Compare sales performance across different states.",
        "Which cities are the top performers?"
    ],
    "COMPARATIVE ANALYSIS": [
        "Compare Technology vs Furniture sales trends.",
        "How does the West region compare to the East in terms of profit?"
    ]
}

for category, qs in queries.items():
    print(f"\n{category}")
    for q in qs:
        print(f"\nQ: {q}")
        answer, metadata = rag_query(q, model=MODEL)
        print(f"A: {answer}")
        print(f"Time: {metadata['context_size']} chars | Type: {metadata['query_type']}\n")


TREND ANALYSIS

Q: Which months show the highest sales? Is there seasonality?
A:  - The top three months with the highest sales are November ($352,461.07), December ($325,293.50), and September ($307,649.95).
- Yes, there is seasonality as sales peak during the last quarter of the year (November to December).
Time: 2024 chars | Type: monthly


Q: How has profit margin changed over time?
A:  - Overall profit margin in 2017 was 12.47%, compared to:
  - 2016: 13.43%
  - 2015: 13.10%
  - 2014: 10.23%
Time: 1808 chars | Type: yearly


CATEGORY ANALYSIS

Q: Which product category generates the most revenue?
A:  The product category that generates the most revenue is Technology, with sales of $836,154.03 in 2017.
Time: 1807 chars | Type: category


Q: What sub-categories have the highest profit margins?
A:  - Sub-Category Art has the highest profit margin at 24.07%
- Sub-Category Accessories has the second highest profit margin at 25.05%
- Sub-Category Appliances has the third highest profit

In [8]:
test_query = "Which months show the highest sales?"
context, n_global, n_grouped, n_raw = retrieve_context(test_query)
print("RETRIEVED CONTEXT:")
print(context)
print("\n" + "="*50)
answer, metadata = rag_query(test_query, model=MODEL)
print(f"ANSWER: {answer}")

RETRIEVED CONTEXT:
GLOBAL SUMMARY: Total Sales $2,297,200.86, Total Profit $286,397.02, Total Orders 9,994, Overall Profit Margin 12.47%
TOP SALES MONTHS: November ($352,461.07), December ($325,293.50), September ($307,649.95)
Year 2014: Sales $484,247.50, Profit $49,543.97, Margin 10.23%
Year 2015: Sales $470,532.51, Profit $61,618.60, Margin 13.10%
Year 2016: Sales $609,205.60, Profit $81,795.17, Margin 13.43%

Central: Sales $501,239.89, Profit $39,706.36, Margin 7.92%
South: Sales $391,721.91, Profit $46,749.43, Margin 11.93%
Sub-Category Accessories: $167,380.32 sales, 25.05% margin, 775 items, 304 discounted
Sub-Category Appliances: $107,532.16 sales, 16.87% margin, 466 items, 195 discounted
Sub-Category Art: $27,118.79 sales, 24.07% margin, 796 items, 298 discounted

Year 2017: Sales $733,215.26, Profit $93,439.27, Margin 12.74%
Furniture: Sales $741,999.80, Profit $18,451.27, Margin 2.49%
Office Supplies: Sales $719,047.03, Profit $122,490.80, Margin 17.04%
Technology: Sales $8

KeyboardInterrupt: 